In [10]:
# Data manipulation
import numpy as np
import pandas as pd

# Preprocessing
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

# Models
from sklearn.linear_model import Ridge
from xgboost import XGBRegressor

# Model selection and evaluation
from sklearn.model_selection import KFold, cross_val_score, GridSearchCV
from sklearn.metrics import r2_score
from sklearn.model_selection import train_test_split
from sklearn.base import clone
# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

1) Load the pre-processed data

In [3]:
# Load preprocessed features
X_train = pd.read_csv("../X_train_median.csv", index_col="id")
X_test = pd.read_csv("../X_test_median.csv", index_col="id")

# Load target values and align with retained patients
y = pd.read_csv("../y_train.csv", index_col="id")["y"]
y_train = y.loc[X_train.index]

# Verify alignment
print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)
print("Indices aligned:", X_train.index.equals(y_train.index))

X_train shape: (1152, 216)
y_train shape: (1152,)
Indices aligned: True


2) Rigdge regression - standard scaler and alpha tuning

In [4]:
# Pipeline: Scaling + Ridge Regression
ridge_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("ridge", Ridge())
])

# Hyperparameter grid
param_grid = {
    "ridge__alpha": np.logspace(-3, 5, 30)
}

# 5-fold cross-validation
cv = KFold(n_splits=5, shuffle=True, random_state=42)

# Grid search
ridge_search = GridSearchCV(
    estimator=ridge_pipeline,
    param_grid=param_grid,
    scoring="r2",
    cv=cv,
    n_jobs=-1
)

ridge_search.fit(X_train, y_train)

# Results
print("Best alpha:", ridge_search.best_params_["ridge__alpha"])
print("Best CV R²:", ridge_search.best_score_)
print("Training R²:", ridge_search.score(X_train, y_train))

Best alpha: 329.0344562312671
Best CV R²: 0.5586985045933207
Training R²: 0.644929111371548


3) XGBoost 
Tuning of : - n_estimtors
            - max_depth
            - learning_rate
            - min_child_weight (contrôle les divisions des arbres pour limiter l'overfitting)
On utilise le même cv (même Kfold que pour ridge)

In [5]:
# XGBoost model
xgb = XGBRegressor(
    objective="reg:squarederror",
    random_state=42,
    n_jobs=1,
    tree_method="hist"
)

# Hyperparameter grid
xgb_param_grid = {
    "n_estimators": [200, 500],
    "max_depth": [2, 3, 5],
    "learning_rate": [0.03, 0.1],
    "min_child_weight": [1, 5]
}

# Grid search with the same CV folds as Ridge
xgb_search = GridSearchCV(
    estimator=xgb,
    param_grid=xgb_param_grid,
    scoring="r2",
    cv=cv,
    n_jobs=-1
)

xgb_search.fit(X_train, y_train)

# Results
print("Best parameters:", xgb_search.best_params_)
print("Best CV R²:", xgb_search.best_score_)
print("Training R²:", xgb_search.score(X_train, y_train))

Best parameters: {'learning_rate': 0.1, 'max_depth': 3, 'min_child_weight': 5, 'n_estimators': 500}
Best CV R²: 0.6352831823031053
Training R²: 0.995566302530086


We now keep the best parameters from above and tune the following to try and improve R2 in validation and reduce the overfitting : 
    - subsample 
    - colsample_bytree
    - reg_lambda
    - reg_alpha

In [6]:
# XGBoost with best parameters from first search
xgb_reg = XGBRegressor(
    objective="reg:squarederror",
    learning_rate=0.1,
    max_depth=3,
    min_child_weight=5,
    n_estimators=500,
    tree_method="hist",
    random_state=42,
    n_jobs=1
)

# Additional hyperparameter grid
reg_param_grid = {
    "subsample": [0.6, 0.8, 1.0],
    "colsample_bytree": [0.6, 0.8, 1.0],
    "reg_lambda": [1, 10, 100],
    "reg_alpha": [0, 1, 10]
}

# Grid search with the same CV folds
xgb_reg_search = GridSearchCV(
    estimator=xgb_reg,
    param_grid=reg_param_grid,
    scoring="r2",
    cv=cv,
    n_jobs=-1
)

xgb_reg_search.fit(X_train, y_train)

# Results
print("Best parameters:", xgb_reg_search.best_params_)
print("Best CV R²:", xgb_reg_search.best_score_)
print("Training R²:", xgb_reg_search.score(X_train, y_train))

Best parameters: {'colsample_bytree': 0.6, 'reg_alpha': 10, 'reg_lambda': 1, 'subsample': 1.0}
Best CV R²: 0.6428402976846954
Training R²: 0.9916922846197237


4) Load kNN sets 

In [7]:
# Load KNN-imputed features
X_train_knn = pd.read_csv("../X_train_knn.csv", index_col="id")

# Align target values
y_train_knn = y.loc[X_train_knn.index]

# Check consistency
print("KNN shape:", X_train_knn.shape)
print("Same patients:", X_train_knn.index.equals(X_train.index))
print("Same features:", X_train_knn.columns.equals(X_train.columns))
print("Labels aligned:", X_train_knn.index.equals(y_train_knn.index))

KNN shape: (1152, 216)
Same patients: True
Same features: True
Labels aligned: True


5) Ridge regression - kNN imputation

In [8]:
# Pipeline: Scaling + Ridge Regression
ridge_knn_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("ridge", Ridge())
])

# Hyperparameter grid
ridge_knn_param_grid = {
    "ridge__alpha": np.logspace(-3, 5, 30)
}

# Grid search with the same CV folds
ridge_knn_search = GridSearchCV(
    estimator=ridge_knn_pipeline,
    param_grid=ridge_knn_param_grid,
    scoring="r2",
    cv=cv,
    n_jobs=-1
)

ridge_knn_search.fit(X_train_knn, y_train_knn)

# Results
print("Best alpha:", ridge_knn_search.best_params_["ridge__alpha"])
print("Best CV R²:", ridge_knn_search.best_score_)
print("Training R²:", ridge_knn_search.score(X_train_knn, y_train_knn))

Best alpha: 174.33288221999874
Best CV R²: 0.5816641439775055
Training R²: 0.6687751350512661


6) XGBoost - kNN Imputation

In [9]:
# XGBoost model
xgb_knn = XGBRegressor(
    objective="reg:squarederror",
    random_state=42,
    n_jobs=1,
    tree_method="hist"
)

# Hyperparameter grid
xgb_knn_param_grid = {
    "n_estimators": [200, 500],
    "max_depth": [2, 3, 5],
    "learning_rate": [0.03, 0.1],
    "min_child_weight": [1, 5]
}

# Grid search with the same CV folds
xgb_knn_search = GridSearchCV(
    estimator=xgb_knn,
    param_grid=xgb_knn_param_grid,
    scoring="r2",
    cv=cv,
    n_jobs=-1
)

xgb_knn_search.fit(X_train_knn, y_train_knn)

# Results
print("Best parameters:", xgb_knn_search.best_params_)
print("Best CV R²:", xgb_knn_search.best_score_)
print("Training R²:", xgb_knn_search.score(X_train_knn, y_train_knn))

Best parameters: {'learning_rate': 0.1, 'max_depth': 3, 'min_child_weight': 5, 'n_estimators': 500}
Best CV R²: 0.6710320206145349
Training R²: 0.9955443122397488


Keep best param from above and add early stopping to try and limit overfit 

In [11]:
# Store results
early_stop_scores = []
best_iterations = []
train_scores = []

for train_idx, val_idx in cv.split(X_train_knn):

    # Outer training and validation folds
    X_tr = X_train_knn.iloc[train_idx]
    X_val = X_train_knn.iloc[val_idx]

    y_tr = y_train_knn.iloc[train_idx]
    y_val = y_train_knn.iloc[val_idx]

    # Inner split for early stopping
    X_fit, X_es, y_fit, y_es = train_test_split(
        X_tr, y_tr,
        test_size=0.2,
        random_state=42
    )

    # XGBoost model
    model = XGBRegressor(
        objective="reg:squarederror",
        n_estimators=2000,
        learning_rate=0.1,
        max_depth=3,
        min_child_weight=5,
        early_stopping_rounds=50,
        eval_metric="rmse",
        tree_method="hist",
        random_state=42,
        n_jobs=1
    )

    # Fit with early stopping
    model.fit(
        X_fit, y_fit,
        eval_set=[(X_es, y_es)],
        verbose=False
    )

    # Evaluate on outer validation fold
    y_pred = model.predict(X_val)

    early_stop_scores.append(r2_score(y_val, y_pred))
    best_iterations.append(model.best_iteration + 1)
    train_scores.append(r2_score(y_fit, model.predict(X_fit)))

# Results
print("Mean CV R²:", np.mean(early_stop_scores))
print("Std CV R²:", np.std(early_stop_scores))
print("Mean Training R²:", np.mean(train_scores))
print("Best number of trees per fold:", best_iterations)
print("Mean best number of trees:", np.mean(best_iterations))

Mean CV R²: 0.6328097253032305
Std CV R²: 0.01784172956074501
Mean Training R²: 0.987482039193749
Best number of trees per fold: [447, 222, 528, 146, 446]
Mean best number of trees: 357.8


Tuning of next set of parameters like in the median imputation case 

In [12]:
# XGBoost with best parameters from first KNN search
xgb_knn_reg = XGBRegressor(
    objective="reg:squarederror",
    learning_rate=0.1,
    max_depth=3,
    min_child_weight=5,
    n_estimators=500,
    tree_method="hist",
    random_state=42,
    n_jobs=1
)

# Regularization hyperparameter grid
knn_reg_param_grid = {
    "subsample": [0.6, 0.8, 1.0],
    "colsample_bytree": [0.6, 0.8, 1.0],
    "reg_lambda": [1, 10, 100],
    "reg_alpha": [0, 1, 10]
}

# Grid search with the same CV folds
xgb_knn_reg_search = GridSearchCV(
    estimator=xgb_knn_reg,
    param_grid=knn_reg_param_grid,
    scoring="r2",
    cv=cv,
    n_jobs=-1
)

xgb_knn_reg_search.fit(X_train_knn, y_train_knn)

# Results
print("Best parameters:", xgb_knn_reg_search.best_params_)
print("Best CV R²:", xgb_knn_reg_search.best_score_)
print("Training R²:", xgb_knn_reg_search.score(X_train_knn, y_train_knn))

Best parameters: {'colsample_bytree': 0.6, 'reg_alpha': 1, 'reg_lambda': 1, 'subsample': 0.8}
Best CV R²: 0.6777864129684112
Training R²: 0.9957579876261943
